# 04 — Land Use Mix (Paris / EUBUCCO)

Computes Shannon entropy of EUBUCCO building subtype distribution per grid cell.
Measures how mixed vs. homogeneous each cell is in terms of building use.

**Replaces:** NYC PLUTO landuse entropy with EUBUCCO subtype entropy.

**Output columns:** `cell_id`, `landuse_entropy`

**Output file:** `csv/Paris/04_land_use_mix.csv`

In [1]:
PARIS_CONFIG = "paris.json"

In [2]:
import pandas as pd
import numpy as np
import geopandas as gpd
import json
import os

with open(PARIS_CONFIG, encoding="utf-8") as f:
    config = json.load(f)

NUTS_CODE   = config["nuts_code"]
CSV_DIR     = config["csv_dir"]
os.makedirs(CSV_DIR, exist_ok=True)

df_grid = pd.read_csv(f"{CSV_DIR}/01_grid_definition.csv", dtype={"cell_id": str})
valid_cells = set(df_grid["cell_id"])

gp       = config["grid_params"]
LAT_MIN  = gp["lat_min"]
LON_MIN  = gp["lon_min"]
LAT_STEP = gp["lat_step"]
LON_STEP = gp["lon_step"]
print(f"Loaded {len(df_grid)} grid cells")

Loaded 120331 grid cells


In [3]:
# ── Stream EUBUCCO ────────────────────────────────────
storage_opts = {
    "anon": True,
    "client_kwargs": {"endpoint_url": "https://s3.eubucco.com"}
}
path = f"s3://eubucco/v0.2/buildings/parquet/nuts_id={NUTS_CODE}/{NUTS_CODE}.parquet"
print("Streaming EUBUCCO...")

gdf = gpd.read_parquet(path, storage_options=storage_opts)
gdf = gdf.to_crs("EPSG:4326")
gdf["latitude"]  = gdf.geometry.centroid.y
gdf["longitude"] = gdf.geometry.centroid.x
gdf = gdf.dropna(subset=["latitude", "longitude", "subtype"])
print(f"Loaded {len(gdf):,} buildings")

Streaming EUBUCCO...


C:\Users\Hani\AppData\Local\Temp\ipykernel_15476\409430191.py:11: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["latitude"]  = gdf.geometry.centroid.y


C:\Users\Hani\AppData\Local\Temp\ipykernel_15476\409430191.py:12: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["longitude"] = gdf.geometry.centroid.x


Loaded 3,594,995 buildings


In [4]:
# ── Assign buildings to cells ─────────────────────────
gdf["grid_row"] = ((gdf["latitude"]  - LAT_MIN) / LAT_STEP).astype(int)
gdf["grid_col"] = ((gdf["longitude"] - LON_MIN) / LON_STEP).astype(int)
gdf["cell_id"]  = "r" + gdf["grid_row"].astype(str).str.zfill(4) + "_c" + gdf["grid_col"].astype(str).str.zfill(4)
gdf = gdf[gdf["cell_id"].isin(valid_cells)].copy()
print(f"Buildings in valid cells: {len(gdf):,}")
print(f"Unique subtypes: {gdf['subtype'].nunique()}")

Buildings in valid cells: 3,505,711
Unique subtypes: 9


In [5]:
# ── Shannon entropy per cell ──────────────────────────
def shannon_entropy(proportions):
    proportions = proportions[proportions > 0]
    if len(proportions) == 0:
        return 0.0
    return -np.sum(proportions * np.log2(proportions))

records = []
for cell_id, group in gdf.groupby("cell_id"):
    subtype_counts = group["subtype"].value_counts()
    proportions    = (subtype_counts / subtype_counts.sum()).values
    records.append({
        "cell_id": cell_id,
        "landuse_entropy": round(shannon_entropy(proportions), 4),
    })

df_mix = pd.DataFrame(records)
df_result = df_grid[["cell_id"]].merge(df_mix, on="cell_id", how="left")
print(f"Computed entropy for {len(df_result)} cells")
print(f"Entropy: mean={df_result['landuse_entropy'].mean():.3f}, "
      f"min={df_result['landuse_entropy'].min():.3f}, "
      f"max={df_result['landuse_entropy'].max():.3f}")

Computed entropy for 120331 cells
Entropy: mean=0.987, min=-0.000, max=2.721


In [6]:
# ── Save output ───────────────────────────────────────
output_path = f"{CSV_DIR}/04_land_use_mix.csv"
df_result.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(df_result)} rows x {df_result.shape[1]} cols)")
df_result.head(10)

Saved: csv/Paris/04_land_use_mix.csv  (120331 rows x 2 cols)


,cell_id,landuse_entropy
0,r0004_c0596,1.3838
1,r0004_c0597,0.9852
2,r0004_c0606,1.3815
3,r0006_c0521,1.5589
4,r0006_c0522,1.0000
5,r0006_c0601,0.8113
6,r0007_c0521,1.0870
7,r0007_c0594,1.5715
8,r0007_c0595,1.8424
9,r0007_c0596,1.3788
